# Week 9 - GenAI Domain Assistant (Part 1)
**Using the Gemini API instead of OpenAI**

**Author**: Muhammad Raheel Ijaz 
**Date**: 8/16/2026

This notebook follows the Lab 9 instructions, but every OpenAI call has been
replaced with the equivalent Google Gemini API call.


## Part 1: Your First API Call
### Task 1.1: Setup Environment

In [1]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Initialize the Gemini client
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

print('Gemini client initialized!')


Gemini client initialized!


### Task 1.2: Make First API Call

In [2]:
# Simple completion (single-turn)
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Say hello and introduce yourself!',
    config=types.GenerateContentConfig(
        max_output_tokens=150,
        temperature=0.7,
    ),
)

# Extract response text
message = response.text
print(message)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Hello! I'm an AI assistant, trained by Google. It's nice to meet you!


**Understanding Parameters:**
- `model`: Which Gemini model to use (e.g. `gemini-2.5-flash`, `gemini-2.5-pro`)
- `contents`: The prompt / conversation history
- `max_output_tokens`: Maximum length of the response
- `temperature`: Creativity (0 = focused, 1 = creative)


## Part 2: Build Basic Chatbot
### Task 2.1: Create Conversation Loop

In [3]:
def chat(history, user_message, model_name='gemini-2.5-flash', system_prompt=None):
    """
    Send the running conversation history + new user message to Gemini
    and return the assistant's reply as text. Also appends both turns
    to 'history' so context carries over (multi-turn memory).
    """
    history.append({'role': 'user', 'parts': [{'text': user_message}]})

    config = types.GenerateContentConfig(
        max_output_tokens=500,
        temperature=0.7,
    )
    if system_prompt:
        config.system_instruction = system_prompt

    response = client.models.generate_content(
        model=model_name,
        contents=history,
        config=config,
    )

    assistant_reply = response.text
    history.append({'role': 'model', 'parts': [{'text': assistant_reply}]})
    return assistant_reply


# Initialize conversation
messages = []

# Conversation loop
print('Chatbot ready! Type "quit" to exit.\n')
while True:
    user_input = input('You: ')
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break

    assistant_response = chat(messages, user_input)
    print(f'Assistant: {assistant_response}\n')


Chatbot ready! Type "quit" to exit.



You:  What can you help me with?


Assistant: As an AI assistant, I can help you with a wide variety of tasks! Think of



You:  quit


Goodbye!


### Task 2.2: Add System Prompt

In [4]:
system_prompt = '''You are a helpful assistant. Be friendly, concise, and
professional. If you don't know something, say so.'''

# Reset conversation with the system prompt applied
messages = []

print('Chatbot ready (with system prompt)! Type "quit" to exit.\n')
while True:
    user_input = input('You: ')
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break

    assistant_response = chat(messages, user_input, system_prompt=system_prompt)
    print(f'Assistant: {assistant_response}\n')


Chatbot ready (with system prompt)! Type "quit" to exit.



You:  What can you help me with?


Assistant: I can help you by answering questions, providing information, generating text, and assisting with a wide range of tasks through natural language. Just let me know what you need!



You:  quit


Goodbye!


**System Prompt:** This sets the AI's personality and behavior. In the Gemini API it is passed as `system_instruction` inside `GenerateContentConfig`, and it stays in effect for the entire conversation.

---
### Next steps
Part 3 (the HR Assistant and Customer Support Bot) is provided as two separate, ready-to-run scripts in this same folder so they're easier to run/test from a terminal:
- `hr_assistant.py`
- `support_assistant.py`

You can also copy their system prompts into this notebook and reuse the `chat()` function above if you prefer to keep everything in one notebook.